# LookBook — Sistema di Gestione degli Elementi (SGE)
### Progetto Python | Settore: Marketplace di abbigliamento usato

**Autore:** Valerio Aquilani
**Corso:** Python — start2impact University

**Contesto:** LookBook è un marketplace dedicato all'abbigliamento di seconda mano.
L'obiettivo di questo notebook è realizzare un **Sistema di Gestione degli Elementi**
che permetta agli utenti di:

- registrare nuovi capi d'abbigliamento nel proprio catalogo personale
- visualizzare tutti i capi registrati
- cercare capi in base a uno o più attributi
- generare statistiche sulla propria collezione
- applicare filtri avanzati con condizioni multiple

Ogni **elemento** della collezione rappresenta un **capo d'abbigliamento** con le sue
caratteristiche specifiche (tipo, colore, taglia, marca, prezzo, condizione, ecc.).


## 1. Definizione dell'elemento: la classe `Capo`

Ogni capo d'abbigliamento è modellato come un oggetto della classe `Capo`, con gli
attributi principali richiesti dal brief (tipo, colore, capo d'abbigliamento) più
alcuni attributi extra coerenti con il dominio LookBook (taglia, marca, prezzo,
condizione, disponibilità swap, data di inserimento, venditore).


In [1]:
from datetime import date

class Capo:
    """Rappresenta un singolo capo d'abbigliamento caricato su LookBook."""

    def __init__(self, nome, tipo, colore, taglia, marca, prezzo,
                 condizione="Buono", disponibile_swap=False,
                 venditore="Anonimo", data_inserimento=None):
        self.nome = nome                      # es. "Giacca di jeans"
        self.tipo = tipo                      # es. "Giacca", "Maglia", "Pantalone"
        self.colore = colore                  # es. "Blu"
        self.taglia = taglia                  # es. "M"
        self.marca = marca                    # es. "Levi's"
        self.prezzo = prezzo                  # prezzo in euro (float)
        self.condizione = condizione          # "Nuovo", "Ottimo", "Buono", "Discreto"
        self.disponibile_swap = disponibile_swap  # True/False -> opzione Swap
        self.venditore = venditore
        self.data_inserimento = data_inserimento or date.today().isoformat()

    def __repr__(self):
        swap = "Sì" if self.disponibile_swap else "No"
        return (f"[{self.tipo}] {self.nome} | Colore: {self.colore} | Taglia: {self.taglia} | "
                f"Marca: {self.marca} | Prezzo: {self.prezzo}€ | Condizione: {self.condizione} | "
                f"Swap: {swap} | Venditore: {self.venditore} | Inserito il: {self.data_inserimento}")

    def to_dict(self):
        """Utile per esportare o confrontare i dati del capo come dizionario."""
        return self.__dict__.copy()


## 2. La collezione

La collezione è rappresentata da una semplice **lista** di oggetti `Capo`.
Tutte le funzioni del sistema lavorano su questa lista, passata come parametro
(così il codice resta riutilizzabile e testabile, invece di dipendere da una
variabile globale nascosta).


In [2]:
# Lista che rappresenta il catalogo/collezione dell'utente
catalogo = []


## 3. Funzionalità di base

### 3.1 Registrazione di un nuovo elemento


In [3]:
def registra_capo(collezione, nome, tipo, colore, taglia, marca, prezzo,
                   condizione="Buono", disponibile_swap=False, venditore="Anonimo"):
    """Crea un nuovo Capo e lo aggiunge alla collezione, con validazione dei dati."""

    # --- Validazione input ---
    campi_obbligatori = {"nome": nome, "tipo": tipo, "colore": colore,
                          "taglia": taglia, "marca": marca}
    for campo, valore in campi_obbligatori.items():
        if not isinstance(valore, str) or not valore.strip():
            print(f"❌ Errore: il campo '{campo}' non può essere vuoto.")
            return None

    if not isinstance(prezzo, (int, float)) or prezzo < 0:
        print("❌ Errore: il prezzo deve essere un numero positivo.")
        return None

    condizioni_valide = ("Nuovo", "Ottimo", "Buono", "Discreto")
    if condizione not in condizioni_valide:
        print(f"❌ Errore: condizione non valida. Valori ammessi: {condizioni_valide}")
        return None

    nuovo_capo = Capo(nome, tipo, colore, taglia, marca, prezzo,
                       condizione, disponibile_swap, venditore)
    collezione.append(nuovo_capo)
    print(f"✅ Capo registrato con successo: {nuovo_capo.nome}")
    return nuovo_capo


### 3.2 Visualizzazione di tutti gli elementi

In [4]:
def visualizza_capi(collezione):
    """Stampa a schermo tutti i capi presenti nella collezione."""
    if not collezione:
        print("La collezione è vuota. Registra un capo per iniziare!")
        return

    print(f"--- Catalogo LookBook ({len(collezione)} capi) ---\n")
    for i, capo in enumerate(collezione, start=1):
        print(f"{i}. {capo}")


### 3.3 Ricerca di elementi

La funzione `cerca_capi` permette di cercare per **uno o più attributi**
contemporaneamente. I criteri vengono passati come `**kwargs`, quindi l'utente
può cercare, ad esempio, solo per colore, solo per tipo, oppure per più
attributi insieme (es. tipo **e** colore **e** taglia).

La ricerca sul testo (nome, tipo, colore, marca, condizione, venditore) è
case-insensitive e per corrispondenza parziale; sul prezzo è una corrispondenza
esatta.


In [5]:
def cerca_capi(collezione, **criteri):
    """
    Cerca i capi che soddisfano TUTTI i criteri passati come parametri.
    Esempio d'uso:
        cerca_capi(catalogo, tipo="Giacca", colore="Blu")
    """
    # --- Validazione: avvisa se un criterio non corrisponde a nessun attributo di Capo ---
    if collezione:
        attributi_validi = set(collezione[0].to_dict().keys())
        for attributo in criteri:
            if attributo not in attributi_validi:
                print(f"⚠️  Attenzione: '{attributo}' non è un attributo valido di un Capo. "
                      f"Attributi disponibili: {sorted(attributi_validi)}")

    risultati = []

    for capo in collezione:
        corrisponde = True
        for attributo, valore_cercato in criteri.items():
            valore_capo = getattr(capo, attributo, None)

            if valore_capo is None:
                corrisponde = False
                break

            # Confronto testuale case-insensitive e parziale
            if isinstance(valore_capo, str) and isinstance(valore_cercato, str):
                if valore_cercato.lower() not in valore_capo.lower():
                    corrisponde = False
                    break
            # Confronto esatto per valori non testuali (prezzo, booleani, ecc.)
            else:
                if valore_capo != valore_cercato:
                    corrisponde = False
                    break

        if corrisponde:
            risultati.append(capo)

    print(f"🔎 Trovati {len(risultati)} capi corrispondenti ai criteri {criteri}\n")
    for capo in risultati:
        print(capo)

    return risultati


## 4. Funzionalità aggiuntive

### 4.1 Statistiche sugli elementi

La funzione `statistiche_capi` fornisce una panoramica della collezione:
- conteggio totale dei capi
- distribuzione dei capi per tipo
- distribuzione dei capi per colore
- prezzo medio, minimo e massimo
- capo più recente e più costoso
- percentuale di capi disponibili allo swap


In [6]:
def statistiche_capi(collezione):
    """Genera e stampa statistiche riassuntive sulla collezione."""
    if not collezione:
        print("Nessuna statistica disponibile: la collezione è vuota.")
        return {}

    totale = len(collezione)

    # Distribuzione per tipo
    distribuzione_tipo = {}
    for capo in collezione:
        distribuzione_tipo[capo.tipo] = distribuzione_tipo.get(capo.tipo, 0) + 1

    # Distribuzione per colore
    distribuzione_colore = {}
    for capo in collezione:
        distribuzione_colore[capo.colore] = distribuzione_colore.get(capo.colore, 0) + 1

    prezzi = [capo.prezzo for capo in collezione]
    prezzo_medio = sum(prezzi) / totale
    prezzo_min = min(prezzi)
    prezzo_max = max(prezzi)

    capo_piu_costoso = max(collezione, key=lambda c: c.prezzo)
    capo_piu_recente = max(collezione, key=lambda c: c.data_inserimento)

    tipo_piu_frequente = max(distribuzione_tipo, key=distribuzione_tipo.get)

    swap_disponibili = sum(1 for capo in collezione if capo.disponibile_swap)
    percentuale_swap = (swap_disponibili / totale) * 100

    # Stampa del report
    print("📊 STATISTICHE COLLEZIONE LOOKBOOK\n" + "-" * 40)
    print(f"Totale capi registrati: {totale}")
    print(f"\nDistribuzione per tipo: {distribuzione_tipo}")
    print(f"Tipo più frequente: {tipo_piu_frequente} ({distribuzione_tipo[tipo_piu_frequente]} capi)")
    print(f"\nDistribuzione per colore: {distribuzione_colore}")
    print(f"\nPrezzo medio: {prezzo_medio:.2f}€")
    print(f"Prezzo minimo: {prezzo_min}€ | Prezzo massimo: {prezzo_max}€")
    print(f"\nCapo più costoso: {capo_piu_costoso.nome} ({capo_piu_costoso.prezzo}€)")
    print(f"Capo più recente: {capo_piu_recente.nome} (inserito il {capo_piu_recente.data_inserimento})")
    print(f"\nCapi disponibili per lo Swap: {swap_disponibili}/{totale} ({percentuale_swap:.1f}%)")

    return {
        "totale": totale,
        "distribuzione_tipo": distribuzione_tipo,
        "distribuzione_colore": distribuzione_colore,
        "prezzo_medio": prezzo_medio,
        "prezzo_min": prezzo_min,
        "prezzo_max": prezzo_max,
        "tipo_piu_frequente": tipo_piu_frequente,
        "percentuale_swap": percentuale_swap,
    }


### 4.2 Filtraggio avanzato

Estende la ricerca base permettendo **condizioni multiple e più complesse**
(non solo uguaglianza), come intervalli di prezzo, presenza di parole chiave,
o combinazioni di condizioni.

La funzione accetta una lista di condizioni, ognuna espressa come una funzione
lambda che riceve il capo e restituisce `True`/`False`. In questo modo l'utente
può comporre filtri arbitrariamente complessi.


In [7]:
def filtro_avanzato(collezione, condizioni):
    """
    Applica una lista di condizioni (funzioni lambda) alla collezione e
    restituisce solo i capi che le soddisfano TUTTE contemporaneamente.

    Esempio d'uso:
        filtro_avanzato(catalogo, [
            lambda c: c.prezzo < 30,
            lambda c: c.tipo == "Giacca",
            lambda c: c.disponibile_swap == True
        ])
    """
    risultati = [
        capo for capo in collezione
        if all(condizione(capo) for condizione in condizioni)
    ]

    print(f"🎯 Filtro avanzato: trovati {len(risultati)} capi che soddisfano tutte le condizioni\n")
    for capo in risultati:
        print(capo)

    return risultati


### 4.3 Eliminazione, modifica e ordinamento (estensioni)

Per rendere il sistema più completo e realmente utilizzabile, aggiungiamo tre
funzioni extra, già anticipate nelle conclusioni del notebook:

- `elimina_capo`: rimuove un capo dalla collezione (per nome esatto o per indice).
- `modifica_capo`: aggiorna uno o più attributi di un capo già registrato.
- `ordina_per_prezzo`: restituisce la collezione ordinata per prezzo, crescente o decrescente.


In [8]:
def elimina_capo(collezione, nome):
    """
    Rimuove dalla collezione il primo capo che corrisponde esattamente al nome
    indicato (case-insensitive). Restituisce True se un capo è stato rimosso,
    False se non è stato trovato nessun capo con quel nome.
    """
    for capo in collezione:
        if capo.nome.lower() == nome.lower():
            collezione.remove(capo)
            print(f"🗑️  Capo eliminato: {capo.nome}")
            return True

    print(f"❌ Nessun capo trovato con nome '{nome}'.")
    return False


In [9]:
def modifica_capo(collezione, nome, **modifiche):
    """
    Modifica uno o più attributi del primo capo trovato con il nome indicato
    (case-insensitive). Gli attributi da modificare vengono passati come kwargs.
    Esempio d'uso:
        modifica_capo(catalogo, "Maglione in lana", prezzo=15.0, disponibile_swap=True)
    """
    for capo in collezione:
        if capo.nome.lower() == nome.lower():
            attributi_validi = capo.to_dict().keys()
            for attributo, nuovo_valore in modifiche.items():
                if attributo not in attributi_validi:
                    print(f"⚠️  Attributo '{attributo}' ignorato: non esiste su Capo.")
                    continue
                setattr(capo, attributo, nuovo_valore)
            print(f"✏️  Capo aggiornato: {capo}")
            return capo

    print(f"❌ Nessun capo trovato con nome '{nome}'.")
    return None


In [10]:
def ordina_per_prezzo(collezione, decrescente=False):
    """Restituisce una nuova lista con i capi ordinati per prezzo."""
    if not collezione:
        print("La collezione è vuota: nulla da ordinare.")
        return []

    ordinati = sorted(collezione, key=lambda c: c.prezzo, reverse=decrescente)

    ordine = "decrescente" if decrescente else "crescente"
    print(f"📈 Capi ordinati per prezzo ({ordine}):\n")
    for capo in ordinati:
        print(capo)

    return ordinati


## 5. Test del sistema

Popoliamo il catalogo con alcuni capi di esempio e proviamo tutte le
funzionalità implementate.


In [11]:
# Registrazione di alcuni capi di esempio
registra_capo(catalogo, "Giacca di jeans vintage", "Giacca", "Blu", "M", "Levi's",
              35.0, condizione="Ottimo", disponibile_swap=True, venditore="Marta")

registra_capo(catalogo, "Maglione in lana", "Maglione", "Verde", "L", "Zara",
              18.5, condizione="Buono", disponibile_swap=False, venditore="Luca")

registra_capo(catalogo, "Pantaloni cargo", "Pantalone", "Beige", "42", "H&M",
              22.0, condizione="Discreto", disponibile_swap=True, venditore="Marta")

registra_capo(catalogo, "Camicia di lino", "Camicia", "Bianco", "M", "Massimo Dutti",
              27.0, condizione="Nuovo", disponibile_swap=False, venditore="Giulia")

registra_capo(catalogo, "Giacca a vento", "Giacca", "Nero", "L", "The North Face",
              45.0, condizione="Ottimo", disponibile_swap=True, venditore="Luca")


✅ Capo registrato con successo: Giacca di jeans vintage
✅ Capo registrato con successo: Maglione in lana
✅ Capo registrato con successo: Pantaloni cargo
✅ Capo registrato con successo: Camicia di lino
✅ Capo registrato con successo: Giacca a vento


[Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16

In [12]:
# 3.2 - Visualizziamo tutti i capi registrati
visualizza_capi(catalogo)


--- Catalogo LookBook (5 capi) ---

1. [Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
2. [Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 18.5€ | Condizione: Buono | Swap: No | Venditore: Luca | Inserito il: 2026-07-16
3. [Pantalone] Pantaloni cargo | Colore: Beige | Taglia: 42 | Marca: H&M | Prezzo: 22.0€ | Condizione: Discreto | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
4. [Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16
5. [Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16


In [13]:
# 3.3 - Ricerca per un singolo attributo (tutti i capi di colore Blu)
cerca_capi(catalogo, colore="Blu")


🔎 Trovati 1 capi corrispondenti ai criteri {'colore': 'Blu'}

[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16


[[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16]

In [14]:
# 3.3 - Ricerca per più attributi contemporaneamente (Giacche disponibili per lo swap)
cerca_capi(catalogo, tipo="Giacca", disponibile_swap=True)


🔎 Trovati 2 capi corrispondenti ai criteri {'tipo': 'Giacca', 'disponibile_swap': True}

[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
[Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16


[[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16,
 [Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16]

In [15]:
# 4.1 - Statistiche sulla collezione
statistiche_capi(catalogo)


📊 STATISTICHE COLLEZIONE LOOKBOOK
----------------------------------------
Totale capi registrati: 5

Distribuzione per tipo: {'Giacca': 2, 'Maglione': 1, 'Pantalone': 1, 'Camicia': 1}
Tipo più frequente: Giacca (2 capi)

Distribuzione per colore: {'Blu': 1, 'Verde': 1, 'Beige': 1, 'Bianco': 1, 'Nero': 1}

Prezzo medio: 29.50€
Prezzo minimo: 18.5€ | Prezzo massimo: 45.0€

Capo più costoso: Giacca a vento (45.0€)
Capo più recente: Giacca di jeans vintage (inserito il 2026-07-16)

Capi disponibili per lo Swap: 3/5 (60.0%)


{'totale': 5,
 'distribuzione_tipo': {'Giacca': 2,
  'Maglione': 1,
  'Pantalone': 1,
  'Camicia': 1},
 'distribuzione_colore': {'Blu': 1,
  'Verde': 1,
  'Beige': 1,
  'Bianco': 1,
  'Nero': 1},
 'prezzo_medio': 29.5,
 'prezzo_min': 18.5,
 'prezzo_max': 45.0,
 'tipo_piu_frequente': 'Giacca',
 'percentuale_swap': 60.0}

In [16]:
# 4.2 - Filtro avanzato: capi sotto i 30€ E disponibili per lo swap
filtro_avanzato(catalogo, [
    lambda c: c.prezzo < 30,
    lambda c: c.disponibile_swap == True
])


🎯 Filtro avanzato: trovati 1 capi che soddisfano tutte le condizioni

[Pantalone] Pantaloni cargo | Colore: Beige | Taglia: 42 | Marca: H&M | Prezzo: 22.0€ | Condizione: Discreto | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16


[[Pantalone] Pantaloni cargo | Colore: Beige | Taglia: 42 | Marca: H&M | Prezzo: 22.0€ | Condizione: Discreto | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16]

In [17]:
# 4.2 - Filtro avanzato: capi Nuovi oppure Ottimi, con prezzo tra 20€ e 40€
filtro_avanzato(catalogo, [
    lambda c: c.condizione in ("Nuovo", "Ottimo"),
    lambda c: 20 <= c.prezzo <= 40
])


🎯 Filtro avanzato: trovati 2 capi che soddisfano tutte le condizioni

[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
[Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16


[[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16,
 [Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16]

### 5.1 Test delle funzioni di eliminazione, modifica e ordinamento

Testiamo le tre nuove funzioni introdotte nel paragrafo 4.3, sul catalogo popolato poco sopra.

In [18]:
# 4.3 - Test: modifica di un capo esistente
modifica_capo(catalogo, "Maglione in lana", prezzo=15.0, disponibile_swap=True)


✏️  Capo aggiornato: [Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 15.0€ | Condizione: Buono | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16


[Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 15.0€ | Condizione: Buono | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16

In [19]:
# 4.3 - Test: ordinamento per prezzo (crescente)
ordina_per_prezzo(catalogo)


📈 Capi ordinati per prezzo (crescente):

[Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 15.0€ | Condizione: Buono | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16
[Pantalone] Pantaloni cargo | Colore: Beige | Taglia: 42 | Marca: H&M | Prezzo: 22.0€ | Condizione: Discreto | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
[Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16
[Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
[Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16


[[Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 15.0€ | Condizione: Buono | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16,
 [Pantalone] Pantaloni cargo | Colore: Beige | Taglia: 42 | Marca: H&M | Prezzo: 22.0€ | Condizione: Discreto | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16,
 [Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16,
 [Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16,
 [Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16]

In [20]:
# 4.3 - Test: eliminazione di un capo
elimina_capo(catalogo, "Pantaloni cargo")
visualizza_capi(catalogo)


🗑️  Capo eliminato: Pantaloni cargo
--- Catalogo LookBook (4 capi) ---

1. [Giacca] Giacca di jeans vintage | Colore: Blu | Taglia: M | Marca: Levi's | Prezzo: 35.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Marta | Inserito il: 2026-07-16
2. [Maglione] Maglione in lana | Colore: Verde | Taglia: L | Marca: Zara | Prezzo: 15.0€ | Condizione: Buono | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16
3. [Camicia] Camicia di lino | Colore: Bianco | Taglia: M | Marca: Massimo Dutti | Prezzo: 27.0€ | Condizione: Nuovo | Swap: No | Venditore: Giulia | Inserito il: 2026-07-16
4. [Giacca] Giacca a vento | Colore: Nero | Taglia: L | Marca: The North Face | Prezzo: 45.0€ | Condizione: Ottimo | Swap: Sì | Venditore: Luca | Inserito il: 2026-07-16


## 6. Conclusioni

Il sistema implementato copre tutte le funzionalità richieste dal brief:

- ✅ **Registrazione** di nuovi elementi tramite la classe `Capo` e `registra_capo()`, con validazione dei dati in ingresso (campi obbligatori non vuoti, prezzo positivo, condizione tra i valori ammessi)
- ✅ **Visualizzazione** completa della collezione con `visualizza_capi()`
- ✅ **Ricerca** per uno o più attributi con `cerca_capi()`, che avvisa l'utente se viene usato un criterio di ricerca non valido
- ✅ **Statistiche** dettagliate (distribuzioni, prezzi, capo più frequente/recente/costoso) con `statistiche_capi()`
- ✅ **Filtraggio avanzato** con condizioni multiple e complesse tramite `filtro_avanzato()`

A queste si aggiungono alcune estensioni pensate per rendere il sistema più completo e realmente utilizzabile:

- ✅ **Eliminazione** di un capo con `elimina_capo()`
- ✅ **Modifica** di uno o più attributi di un capo esistente con `modifica_capo()`
- ✅ **Ordinamento** della collezione per prezzo con `ordina_per_prezzo()`

Il codice è pensato per essere facilmente estendibile: si possono aggiungere nuovi
attributi alla classe `Capo` (es. materiale, immagine, stato di vendita) o nuove
funzioni (es. registrazione interattiva da input utente, esportazione su file CSV/JSON)
senza dover riscrivere la logica esistente.
